# CatBoost + HGB Ensemble v2

## Versiyon notları (v1 → v2)

| Alan | v1 | v2 |
|---|---|---|
| Text model | TF-IDF + Ridge OOF | TF-IDF + Ridge OOF (korundu) |
| Ana model | Sadece CatBoost | CatBoost + HGB-1 + HGB-2 |
| Feature engineering | ~60 feature | +15 yeni feature |
| Blend | text_weight=0 en iyi | OOF üzerinde optimize |
| Submission | CatBoost tek | Ridge meta-ensemble |

**v1 OOF RMSE: 8.7433**


In [9]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from scipy import sparse

from catboost import CatBoostRegressor

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

RANDOM_STATE = 42
N_SPLITS = 5

# ── Yolları kendi ortamınıza göre ayarlayın ──
TRAIN_PATH = 'train.csv'
TEST_PATH  = 'test_x.csv'

TARGET = 'career_success_score'
ID_COL = 'student_id'

RESULTS_DIR = Path('results4')
RESULTS_DIR.mkdir(exist_ok=True)

RUN_NAME = 'catboost_ensemble_v2'

In [10]:
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print(f'Train shape: {train_df.shape}')
print(f'Test shape : {test_df.shape}')
print(f'Target min/max: {train_df[TARGET].min()} {train_df[TARGET].max()}')

y = train_df[TARGET]

Train shape: (10000, 47)
Test shape : (10000, 46)
Target min/max: 0.0 100.0


In [11]:
# ─────────────────────────────────────────────
# 1. EKSİK DEĞER BAYRAKLARI
# ─────────────────────────────────────────────
MISSING_FLAG_COLS = [
    'english_exam_score',
    'internship_duration_months',
    'portfolio_score',
    'github_avg_stars',
    'open_source_contribution_count',
    'linkedin_profile_score',
    'hr_interview_score',
]

def add_missing_flags(df):
    df = df.copy()
    for col in MISSING_FLAG_COLS:
        if col in df.columns:
            df[f'{col}_was_missing'] = df[col].isna().astype(int)
    return df

train_df = add_missing_flags(train_df)
test_df  = add_missing_flags(test_df)

In [12]:
# ─────────────────────────────────────────────
# 2. EKSİK DEĞER DOLDURMA
# ─────────────────────────────────────────────
def fill_missing_values(train, test):
    train = train.copy()
    test  = test.copy()

    # Median ile doldur (train'den hesapla, test'e uygula)
    median_cols = [
        'english_exam_score', 'internship_duration_months',
        'portfolio_score', 'github_avg_stars',
        'open_source_contribution_count', 'linkedin_profile_score',
        'hr_interview_score',
    ]
    for col in median_cols:
        if col in train.columns:
            med = train[col].median()
            train[col] = train[col].fillna(med)
            test[col]  = test[col].fillna(med)

    # Text sütunu boş string
    train['mentor_feedback_text'] = train['mentor_feedback_text'].fillna('')
    test['mentor_feedback_text']  = test['mentor_feedback_text'].fillna('')

    return train, test

train_df, test_df = fill_missing_values(train_df, test_df)
print('Missing after fill — train:', train_df.drop(columns=[TARGET]).isna().sum().sum())
print('Missing after fill — test :', test_df.isna().sum().sum())

Missing after fill — train: 0
Missing after fill — test : 0


In [13]:
# ─────────────────────────────────────────────
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────
TECH_COLS = [
    'coding_score', 'problem_solving_score', 'data_structures_score',
    'sql_score', 'machine_learning_score', 'backend_score',
    'frontend_score', 'cloud_score', 'devops_score',
]

SOFT_COLS = [
    'communication_score', 'teamwork_score', 'leadership_score',
    'presentation_score', 'linkedin_profile_score', 'cv_quality_score',
    'hr_interview_score',
]

ROLE_SKILL_MAP = {
    'Backend Developer':         'backend_score',
    'Frontend Developer':        'frontend_score',
    'DevOps Engineer':           'devops_score',
    'Machine Learning Engineer': 'machine_learning_score',
    'Data Scientist':            'machine_learning_score',
    'Cloud Engineer':            'cloud_score',
    'Software Developer':        'coding_score',
    'Full Stack Developer':      'coding_score',
    'Data Engineer':             'sql_score',
    'Mobile Developer':          'frontend_score',
    'Embedded Systems':          'coding_score',
    'Data Analyst':              'sql_score',
    'Product Analyst':           'machine_learning_score',
}

TIER_MAP = {'Tier 1': 4, 'Tier 2': 3, 'Tier 3': 2, 'Tier 4': 1}

NLP_TECH_KW = [
    'makine öğrenimi', 'veri yapıları', 'veri bilimi', 'veri analizi',
    'backend', 'frontend', 'bulut', 'yazılım', 'açık kaynak',
    'github', 'sql', 'yapay zeka', 'devops', 'cloud', 'kodlama',
    'problem çözme', 'algoritma', 'mobil', 'siber güvenlik', 'data',
    'analist', 'mühendis', 'developer',
]

NLP_SOFT_KW = [
    'iletişim', 'takım çalışması', 'mülakat', 'proje', 'takım',
    'liderlik', 'sunum', 'ekip', 'zaman yönetimi',
    'analitik düşünme', 'yönetim', 'ikna',
]

NLP_POS = [
    'etkileyici', 'güçlü', 'dikkat çekici', 'dikkat çekiyor',
    'dikkat çeken', 'sağlam temel', 'yüksek', 'başarılı',
    'harika', 'potansiyel', 'iyi seviyede', 'mükemmel',
    'tatmin edici', 'beklentileri aşıyor', 'yetkin', 'olağanüstü',
    'uzman', 'proaktif', 'yaratıcı', 'lider',
]

NLP_NEG = [
    'faydalı olacaktır', 'fazla pratik', 'deneyim kazanması',
    'pratik yapması', 'geliştirmeli', 'eksik', 'odaklanmalı',
    'çalışmalı', 'yetersiz', 'zayıf', 'beklentilerin altında',
    'artırmalı', 'ihtiyacı var', 'gerekiyor', 'gelişmeli',
    'çalışması faydalı',
]

NLP_CONTRAST = [
    'ancak', 'fakat', 'rağmen', 'gösterse de',
    'bununla birlikte', 'yine de', 'yalnız',
]


def add_features(df):
    df = df.copy()

    # ── Zaman ──
    df['years_since_graduation'] = (
        df['application_year'] - df['graduation_year']
    ).clip(0)
    df['age_at_graduation'] = df['age'] - df['years_since_graduation']

    # ── Üniversite tier ──
    df['university_tier_rank'] = df['university_tier'].map(TIER_MAP).fillna(1).astype(int)

    # ── Teknik beceri istatistikleri ──
    df['technical_skill_mean']  = df[TECH_COLS].mean(axis=1)
    df['technical_skill_std']   = df[TECH_COLS].std(axis=1)
    df['technical_skill_min']   = df[TECH_COLS].min(axis=1)
    df['technical_skill_max']   = df[TECH_COLS].max(axis=1)
    df['technical_skill_range'] = df['technical_skill_max'] - df['technical_skill_min']

    # Alt-domain ortalamalar
    df['data_ai_score']             = df[['sql_score', 'machine_learning_score']].mean(axis=1)
    df['software_engineering_score']= df[['coding_score', 'backend_score', 'frontend_score']].mean(axis=1)
    df['cloud_devops_score']        = df[['cloud_score', 'devops_score']].mean(axis=1)
    df['core_cs_score']             = df[['coding_score', 'problem_solving_score', 'data_structures_score']].mean(axis=1)

    # Role-skill fit
    df['role_skill_fit_score'] = df.apply(
        lambda r: r.get(ROLE_SKILL_MAP.get(r['target_role'], 'coding_score'), r['technical_skill_mean']),
        axis=1
    )
    df['role_fit_minus_technical_mean'] = df['role_skill_fit_score'] - df['technical_skill_mean']

    # ── Soft beceriler ──
    df['soft_skill_mean']               = df[SOFT_COLS].mean(axis=1)
    df['soft_skill_std']                = df[SOFT_COLS].std(axis=1)
    df['communication_presentation_mean'] = df[['communication_score', 'presentation_score']].mean(axis=1)

    # ── Proje / portföy / GitHub ──
    df['project_count_total'] = df['real_client_project_count'] + df['freelance_project_count']
    df['portfolio_project_quality_mean'] = df[['portfolio_score', 'project_quality_score', 'cv_quality_score']].mean(axis=1)
    df['portfolio_project_quality_interaction'] = df['portfolio_score'] * df['project_quality_score'] / 100
    df['project_quality_x_real_client_log'] = df['project_quality_score'] * np.log1p(df['real_client_project_count'])
    df['project_quality_x_project_count_log'] = df['project_quality_score'] * np.log1p(df['project_count_total'])

    df['github_activity_log'] = np.log1p(df['github_repo_count'] * df['github_avg_stars'])
    df['github_impact_log'] = (
        np.log1p(df['github_repo_count'])
        + np.log1p(df['github_avg_stars'])
        + np.log1p(df['open_source_contribution_count'])
    )
    df['github_per_repo_quality'] = df['github_avg_stars'] / (df['github_repo_count'] + 1)
    df['github_open_source_intensity'] = df['open_source_contribution_count'] / (df['github_repo_count'] + 1)

    df['profile_visibility_mean'] = df[['linkedin_profile_score', 'portfolio_score', 'cv_quality_score']].mean(axis=1)
    df['portfolio_x_linkedin'] = df['portfolio_score'] * df['linkedin_profile_score'] / 100

    # ── Deneyim / hackathon / öğrenme ──
    df['experience_count_total'] = (
        df['real_client_project_count'] + df['internship_count']
        + df['freelance_project_count'] + df['hackathon_count']
        + df['certification_count'] + df['bootcamp_count']
    )
    df['experience_count_log'] = np.log1p(df['experience_count_total'])
    df['total_internship_months'] = df['internship_count'] * df['internship_duration_months']
    df['internship_months_per_internship'] = df['internship_duration_months'] / (df['internship_count'] + 1)
    df['internship_experience_log'] = np.log1p(df['internship_count']) + np.log1p(df['internship_duration_months'])

    df['hackathon_award_rate'] = df['hackathon_awards'] / (df['hackathon_count'] + 1)
    df['hackathon_activity_log'] = np.log1p(df['hackathon_count']) + np.log1p(df['hackathon_awards'])

    df['learning_count_total'] = df['certification_count'] + df['bootcamp_count']
    df['learning_activity_log'] = np.log1p(df['learning_count_total'])
    df['cert_per_year'] = df['certification_count'] / (df['years_since_graduation'] + 1).clip(1)

    # Ağırlıklı deneyim skoru (sklearn kodundan)
    df['experience_score_weighted'] = (
        df['internship_count'] * 10
        + df['real_client_project_count'] * 8
        + df['freelance_project_count'] * 5
        + df['hackathon_count'] * 3
        + df['hackathon_awards'] * 6
    )

    # ── Mülakat / başvuru hunisi ──
    df['interview_conversion_rate'] = df['interviews_attended'] / (df['applications_sent'] + 1)
    df['applications_without_interview'] = df['applications_sent'] - df['interviews_attended']
    df['log_applications_sent'] = np.log1p(df['applications_sent'])
    df['log_interviews_attended'] = np.log1p(df['interviews_attended'])

    df['interview_score_mean'] = df[['technical_interview_score', 'hr_interview_score']].mean(axis=1)
    df['interview_score_gap']  = df['technical_interview_score'] - df['hr_interview_score']
    df['interview_total']      = df['technical_interview_score'] + df['hr_interview_score']

    df['technical_interview_x_conversion'] = df['technical_interview_score'] * df['interview_conversion_rate']
    df['interview_mean_x_conversion']       = df['interview_score_mean'] * df['interview_conversion_rate']

    # ── Kontrollü etkileşimler ──
    df['technical_x_project_quality'] = df['technical_skill_mean'] * df['project_quality_score'] / 100
    df['technical_x_portfolio']       = df['technical_skill_mean'] * df['portfolio_score'] / 100
    df['role_fit_x_project_quality']  = df['role_skill_fit_score'] * df['project_quality_score'] / 100
    df['soft_x_technical_interview']  = df['soft_skill_mean'] * df['technical_interview_score'] / 100
    df['soft_x_interview_mean']       = df['soft_skill_mean'] * df['interview_score_mean'] / 100
    df['profile_x_project_quality']   = df['profile_visibility_mean'] * df['project_quality_score'] / 100

    # ── Akademik (sklearn kodundan gelen fikir) ──
    df['academic_score'] = (
        df['cgpa'] * 20
        + df['attendance_rate'] * 0.3
        - df['failed_courses_count'] * 5
    )
    df['cgpa_attendance'] = df['cgpa'] * df['attendance_rate'] / 100

    # ── Basit NLP keyword feature'ları ──
    feedback_lower = df['mentor_feedback_text'].fillna('').astype(str).str.lower()
    df['mentor_feedback_len']       = feedback_lower.str.len()
    df['mentor_feedback_word_count'] = feedback_lower.str.split().str.len()

    df['nlp_tech_keyword_count']     = feedback_lower.apply(lambda t: sum(k in t for k in NLP_TECH_KW))
    df['nlp_soft_keyword_count']     = feedback_lower.apply(lambda t: sum(k in t for k in NLP_SOFT_KW))
    df['nlp_positive_signal_count']  = feedback_lower.apply(lambda t: sum(k in t for k in NLP_POS))
    df['nlp_negative_signal_count']  = feedback_lower.apply(lambda t: sum(k in t for k in NLP_NEG))
    df['nlp_has_contrast']           = feedback_lower.apply(lambda t: int(any(w in t for w in NLP_CONTRAST)))
    df['nlp_sentiment_balance']      = df['nlp_positive_signal_count'] - df['nlp_negative_signal_count']

    # ── Composite skorlar (sklearn kodundan) ──
    df['portfolio_composite'] = (
        df['portfolio_score'] * 0.4
        + df['cv_quality_score'] * 0.3
        + df['linkedin_profile_score'] * 0.3
    )
    df['overall_composite'] = (
        df['technical_skill_mean'] * 0.3
        + df['soft_skill_mean'] * 0.15
        + df['academic_score'] * 0.01
        + df['experience_score_weighted'] * 0.3
        + df['portfolio_composite'] * 0.15
        + df['nlp_sentiment_balance'] * 2
    )

    df = df.replace([np.inf, -np.inf], np.nan)
    return df


train_fe = add_features(train_df)
test_fe  = add_features(test_df)

print('Train shape after features:', train_fe.shape)
print('Test  shape after features:', test_fe.shape)

Train shape after features: (10000, 120)
Test  shape after features: (10000, 119)


In [14]:
# ─────────────────────────────────────────────
# 4. TEXT OOF FEATURE (TF-IDF + Ridge)
# ─────────────────────────────────────────────
def build_text_oof_features(train_df, test_df, target_col, text_col='mentor_feedback_text', seed=RANDOM_STATE):
    train_text = train_df[text_col].fillna('').astype(str)
    test_text  = test_df[text_col].fillna('').astype(str)
    y_text     = train_df[target_col].values

    word_vec = TfidfVectorizer(
        analyzer='word', ngram_range=(1, 3),
        min_df=3, max_features=40000,
        sublinear_tf=True, lowercase=True
    )
    char_vec = TfidfVectorizer(
        analyzer='char_wb', ngram_range=(3, 5),
        min_df=3, max_features=40000,
        sublinear_tf=True, lowercase=True
    )

    print('Fitting word TF-IDF...')
    X_word      = word_vec.fit_transform(train_text)
    X_test_word = word_vec.transform(test_text)

    print('Fitting char TF-IDF...')
    X_char      = char_vec.fit_transform(train_text)
    X_test_char = char_vec.transform(test_text)

    X_text      = sparse.hstack([X_word, X_char]).tocsr()
    X_test_text = sparse.hstack([X_test_word, X_test_char]).tocsr()

    print(f'Text train matrix: {X_text.shape}')

    kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    text_oof  = np.zeros(len(train_df))
    text_test = np.zeros(len(test_df))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_text), start=1):
        model = Ridge(alpha=20.0, solver='lsqr')
        model.fit(X_text[tr_idx], y_text[tr_idx])

        val_pred = model.predict(X_text[val_idx])
        text_oof[val_idx] = val_pred

        fold_rmse = mean_squared_error(y_text[val_idx], val_pred) ** 0.5
        fold_scores.append(fold_rmse)
        print(f'  Text Fold {fold} RMSE: {fold_rmse:.5f}')

        text_test += model.predict(X_test_text) / N_SPLITS

    text_oof  = np.clip(text_oof,  0, 100)
    text_test = np.clip(text_test, 0, 100)

    text_rmse = mean_squared_error(y_text, text_oof) ** 0.5
    print(f'\nText OOF RMSE: {text_rmse:.5f}')

    return {
        'text_oof': text_oof,
        'text_test': text_test,
        'text_oof_rmse': text_rmse,
    }


text_result = build_text_oof_features(train_df, test_df, target_col=TARGET)

global_target_mean = train_df[TARGET].mean()

# Text OOF'tan türetilen feature'lar
for df in [train_fe, test_fe]:
    is_train = (df is train_fe)
    arr = text_result['text_oof'] if is_train else text_result['text_test']
    df['text_model_pred']          = arr
    df['text_model_pred_centered'] = arr - global_target_mean
    df['text_model_pred_rank']     = pd.Series(arr).rank(pct=True).values

train_fe['text_model_pred_x_role_fit'] = train_fe['text_model_pred'] * train_fe['role_skill_fit_score'] / 100
test_fe['text_model_pred_x_role_fit']  = test_fe['text_model_pred'] * test_fe['role_skill_fit_score'] / 100

print(f'Train shape after text features: {train_fe.shape}')
print(f'Test  shape after text features: {test_fe.shape}')

Fitting word TF-IDF...
Fitting char TF-IDF...
Text train matrix: (10000, 52146)
  Text Fold 1 RMSE: 12.52765
  Text Fold 2 RMSE: 12.52005
  Text Fold 3 RMSE: 12.15592
  Text Fold 4 RMSE: 12.06349
  Text Fold 5 RMSE: 12.35794

Text OOF RMSE: 12.32596
Train shape after text features: (10000, 124)
Test  shape after text features: (10000, 123)


In [15]:
# ─────────────────────────────────────────────
# 5. MODEL MATRİSİ HAZIRLAMA
# ─────────────────────────────────────────────
DROP_COLS = [ID_COL, TARGET, 'mentor_feedback_text']
CAT_COLS  = ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']

feature_cols = [c for c in train_fe.columns if c not in DROP_COLS]

X      = train_fe[feature_cols].copy()
X_test = test_fe[feature_cols].copy()

# Kategorikler string olmalı (CatBoost için)
for col in CAT_COLS:
    if col in X.columns:
        X[col]      = X[col].astype(str)
        X_test[col] = X_test[col].astype(str)

cat_feature_indices = [X.columns.tolist().index(c) for c in CAT_COLS if c in X.columns]

# HGB için kategorikleri encode et (ordinal)
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X_hgb      = X.copy()
X_test_hgb = X_test.copy()
X_hgb[CAT_COLS]      = oe.fit_transform(X[CAT_COLS].astype(str))
X_test_hgb[CAT_COLS] = oe.transform(X_test[CAT_COLS].astype(str))
X_hgb      = X_hgb.values.astype(np.float64)
X_test_hgb = X_test_hgb.values.astype(np.float64)

print(f'Feature sayısı: {len(feature_cols)}')
print(f'Categorical features: {CAT_COLS}')

Feature sayısı: 121
Categorical features: ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']


In [16]:
# ─────────────────────────────────────────────
# 6. CATBOOST CV
# ─────────────────────────────────────────────
cat_params = {
    'loss_function':      'RMSE',
    'eval_metric':        'RMSE',
    'iterations':         2500,
    'learning_rate':      0.02,
    'depth':              6,
    'l2_leaf_reg':        8,
    'min_data_in_leaf':   20,
    'random_strength':    1.0,
    'bagging_temperature':0.5,
    'random_seed':        RANDOM_STATE,
    'od_type':            'Iter',
    'od_wait':            200,
    'verbose':            500,
    'allow_writing_files': False,
}


def run_catboost_cv(params, X, y, X_test, cat_feature_indices, n_splits=N_SPLITS, seed=RANDOM_STATE):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_preds  = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X, y), start=1):
        print(f'\n========== CatBoost Fold {fold} ==========')
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr,
                  cat_features=cat_feature_indices,
                  eval_set=(X_val, y_val),
                  use_best_model=True)

        val_pred = model.predict(X_val)
        oof_preds[val_idx] = val_pred

        fold_rmse = mean_squared_error(y_val, val_pred) ** 0.5
        fold_scores.append(fold_rmse)
        print(f'Fold {fold} RMSE: {fold_rmse:.5f}')

        test_preds += model.predict(X_test) / n_splits

    oof_rmse = mean_squared_error(y, oof_preds) ** 0.5
    print(f'\n=== CatBoost OOF RMSE: {oof_rmse:.5f} ===')
    return {'oof_preds': oof_preds, 'test_preds': test_preds, 'oof_rmse': oof_rmse}


cb_result = run_catboost_cv(cat_params, X, y, X_test, cat_feature_indices)


========== CatBoost Fold 1 ==========
0:	learn: 15.0484982	test: 15.0316672	best: 15.0316672 (0)	total: 34.7ms	remaining: 1m 26s
500:	learn: 8.2419652	test: 9.0048625	best: 9.0048625 (500)	total: 15.1s	remaining: 1m
1000:	learn: 7.5130539	test: 8.8943854	best: 8.8943854 (1000)	total: 27.5s	remaining: 41.2s
1500:	learn: 6.8968876	test: 8.8680931	best: 8.8678232 (1498)	total: 37.9s	remaining: 25.2s
2000:	learn: 6.3854949	test: 8.8606877	best: 8.8579029 (1944)	total: 48s	remaining: 12s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 8.857902904
bestIteration = 1944

Shrink model to first 1945 iterations.
Fold 1 RMSE: 8.85790

========== CatBoost Fold 2 ==========
0:	learn: 14.9973473	test: 15.2417463	best: 15.2417463 (0)	total: 16.7ms	remaining: 41.7s
500:	learn: 8.2078965	test: 9.2024114	best: 9.2024114 (500)	total: 9.87s	remaining: 39.4s
1000:	learn: 7.4694325	test: 9.0899549	best: 9.0899549 (1000)	total: 19.5s	remaining: 29.2s
1500:	learn: 6.8993990	test: 9.0616013	

In [17]:
# ─────────────────────────────────────────────
# 7. HGB CV  (NaN native — imputation gerekmez)
# ─────────────────────────────────────────────
HGB_CONFIGS = [
    dict(max_iter=1500, learning_rate=0.04, max_leaf_nodes=47, min_samples_leaf=20,
         l2_regularization=0.1, random_state=42),
    dict(max_iter=1200, learning_rate=0.025, max_leaf_nodes=31, min_samples_leaf=30,
         l2_regularization=0.5, random_state=123),
    dict(max_iter=1000, learning_rate=0.06, max_leaf_nodes=63, min_samples_leaf=15,
         l2_regularization=0.05, random_state=7),
]


def run_hgb_cv(config, X_np, y, X_test_np, n_splits=N_SPLITS, seed=RANDOM_STATE, name='HGB'):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof_preds  = np.zeros(len(y))
    test_preds = np.zeros(len(X_test_np))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_np, y), start=1):
        model = HistGradientBoostingRegressor(
            early_stopping=True, validation_fraction=0.1,
            n_iter_no_change=40, **config
        )
        model.fit(X_np[tr_idx], y.values[tr_idx])
        val_pred = model.predict(X_np[val_idx])
        oof_preds[val_idx] = val_pred

        fold_rmse = mean_squared_error(y.values[val_idx], val_pred) ** 0.5
        fold_scores.append(fold_rmse)

        test_preds += model.predict(X_test_np) / n_splits

    oof_rmse = mean_squared_error(y.values, oof_preds) ** 0.5
    print(f'{name} OOF RMSE: {oof_rmse:.5f}  (folds: {[round(s,4) for s in fold_scores]})')
    return {'oof_preds': oof_preds, 'test_preds': test_preds, 'oof_rmse': oof_rmse}


hgb_results = []
for i, cfg in enumerate(HGB_CONFIGS, start=1):
    print(f'\n--- HGB-{i} ---')
    res = run_hgb_cv(cfg, X_hgb, y, X_test_hgb, name=f'HGB-{i}')
    hgb_results.append(res)


--- HGB-1 ---
HGB-1 OOF RMSE: 9.00676  (folds: [9.1029, 9.2733, 8.796, 8.7303, 9.1194])

--- HGB-2 ---
HGB-2 OOF RMSE: 8.96513  (folds: [9.0585, 9.255, 8.6731, 8.6811, 9.1416])

--- HGB-3 ---
HGB-3 OOF RMSE: 9.05100  (folds: [9.1585, 9.3245, 8.7415, 8.8137, 9.2023])


In [18]:
# ─────────────────────────────────────────────
# 8. OOF ÜZERİNDE OPTİMAL BLEND (Ridge meta)
# ─────────────────────────────────────────────
# OOF tahminleri bir araya getir
oof_stack = np.column_stack([
    cb_result['oof_preds'],
    hgb_results[0]['oof_preds'],
    hgb_results[1]['oof_preds'],
    hgb_results[2]['oof_preds'],
])

test_stack = np.column_stack([
    cb_result['test_preds'],
    hgb_results[0]['test_preds'],
    hgb_results[1]['test_preds'],
    hgb_results[2]['test_preds'],
])

# Ridge meta-learner (non-negative, positive_only için clip)
meta = Ridge(alpha=5.0, fit_intercept=True)
meta.fit(oof_stack, y)
meta_oof_pred = np.clip(meta.predict(oof_stack), 0, 100)

meta_oof_rmse = mean_squared_error(y, meta_oof_pred) ** 0.5
print(f'\n=== Meta Ridge OOF RMSE: {meta_oof_rmse:.5f} ===')
print(f'Meta weights: CB={meta.coef_[0]:.3f}, HGB1={meta.coef_[1]:.3f}, HGB2={meta.coef_[2]:.3f}, HGB3={meta.coef_[3]:.3f}')

# Basit ortalama ile karşılaştır
avg_oof = np.clip(oof_stack.mean(axis=1), 0, 100)
avg_rmse = mean_squared_error(y, avg_oof) ** 0.5
print(f'Simple average OOF RMSE: {avg_rmse:.5f}')

# En iyi seçim
if meta_oof_rmse < avg_rmse:
    print('→ Meta Ridge daha iyi, onu kullanıyoruz.')
    final_test_preds = np.clip(meta.predict(test_stack), 0, 100)
    best_oof_rmse = meta_oof_rmse
else:
    print('→ Basit ortalama daha iyi, onu kullanıyoruz.')
    final_test_preds = np.clip(test_stack.mean(axis=1), 0, 100)
    best_oof_rmse = avg_rmse

print(f'\n✅ Final OOF RMSE: {best_oof_rmse:.5f}  (v1: 8.74332)')


=== Meta Ridge OOF RMSE: 8.75023 ===
Meta weights: CB=0.938, HGB1=0.025, HGB2=0.005, HGB3=0.045
Simple average OOF RMSE: 8.84924
→ Meta Ridge daha iyi, onu kullanıyoruz.

✅ Final OOF RMSE: 8.75023  (v1: 8.74332)


In [19]:
# ─────────────────────────────────────────────
# 9. SUBMISSION
# ─────────────────────────────────────────────
submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: np.round(final_test_preds, 4)
})

sub_path = RESULTS_DIR / f'{RUN_NAME}_submission.csv'
submission.to_csv(sub_path, index=False)

print(f'Saved: {sub_path}')
print(f'Shape: {submission.shape}')
display(submission.head(10))
display(submission.describe())

# Metrikleri kaydet
metrics = pd.DataFrame([{
    'run_name':     RUN_NAME,
    'oof_rmse':     best_oof_rmse,
    'cb_oof_rmse':  cb_result['oof_rmse'],
    'hgb1_oof_rmse': hgb_results[0]['oof_rmse'],
    'hgb2_oof_rmse': hgb_results[1]['oof_rmse'],
    'hgb3_oof_rmse': hgb_results[2]['oof_rmse'],
    'text_oof_rmse': text_result['text_oof_rmse'],
}])

metrics_path = RESULTS_DIR / f'{RUN_NAME}_metrics.csv'
metrics.to_csv(metrics_path, index=False)

print(f'\nMetrics saved: {metrics_path}')
display(metrics)

Saved: results4\catboost_ensemble_v2_submission.csv
Shape: (10000, 2)


,student_id,career_success_score
0,STU_010001,58.3813
1,STU_010002,78.3139
2,STU_010003,74.2943
3,STU_010004,96.2588
4,STU_010005,78.3838
5,STU_010006,74.8514
6,STU_010007,76.5822
7,STU_010008,75.6197
8,STU_010009,71.5417
9,STU_010010,55.0909


,career_success_score
count,10000.000000
mean,76.159929
std,13.088074
min,34.347100
25%,66.743050
50%,76.006550
75%,86.049575
max,100.000000



Metrics saved: results4\catboost_ensemble_v2_metrics.csv


,run_name,oof_rmse,cb_oof_rmse,hgb1_oof_rmse,hgb2_oof_rmse,hgb3_oof_rmse,text_oof_rmse
0,catboost_ensemble_v2,8.750231,8.760192,9.006756,8.965126,9.051003,12.325955
